# thermo-agent: Hugging Face -> Kaggle dataset bridge

Downloads `Reza-Nadimi/thermo-agent-corpora` with the private `env` dataset and
publishes it as the single Kaggle dataset `rezanadimi/thermo-agent-data`.
No local data upload; no Kaggle Secrets are required.

In [ ]:
import os
import pathlib

ENV_PATH = None
for candidate in pathlib.Path("/kaggle/input").rglob("*"):
    if candidate.is_file() and candidate.name in ("env", ".env"):
        ENV_PATH = candidate
        break
if ENV_PATH is None:
    raise SystemExit("attach the private dataset rezanadimi/thermo-agent-env (file 'env')")

for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
    if "=" in line and not line.strip().startswith("#"):
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
for key in ("HF_TOKEN", "KAGGLE_API_TOKEN"):
    if not os.environ.get(key):
        raise SystemExit(f"{key} missing from attached env file")
print(f"credentials loaded from {ENV_PATH.name}; values redacted", flush=True)

In [ ]:
from huggingface_hub import snapshot_download
import time

HF_REPO = "Reza-Nadimi/thermo-agent-corpora"
OUT = pathlib.Path("/kaggle/working/hf_corpora")
OUT.mkdir(parents=True, exist_ok=True)
last_error = None
for attempt in range(3):
    try:
        snapshot_download(repo_id=HF_REPO, repo_type="dataset", token=os.environ["HF_TOKEN"],
                          local_dir=str(OUT))
        last_error = None
        break
    except Exception as exc:
        last_error = exc
        print(f"HF snapshot attempt {attempt + 1} failed: {type(exc).__name__}", flush=True)
        time.sleep(10)
if last_error is not None:
    raise last_error
files = sorted(p for p in OUT.rglob("*") if p.is_file())
print(f"downloaded {len(files)} files from {HF_REPO}", flush=True)
assert any("native_pilot_gpt-6-luna" in str(p) for p in files), "pilot shard missing from HF snapshot"


In [ ]:
import json
import shutil
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)

# Kaggle's CLI archives or skips directories, so publish a FLAT dataset: shard files,
# corpus zips and the code bundle all live at the dataset root. The hosted runner
# rglobs /kaggle/input for shard names and kaggle_bundle.zip, so nesting is not needed.
UPLOAD = pathlib.Path("/kaggle/working/upload_data")
if UPLOAD.exists():
    shutil.rmtree(UPLOAD)
UPLOAD.mkdir(parents=True)

copied = []
prompt_dir = OUT / "data" / "frontier_tools_cohort_prompts"
for src in sorted(prompt_dir.iterdir()):
    if src.is_file():
        shutil.copy2(src, UPLOAD / src.name)
        copied.append(src.name)
assert "prompts_native_pilot_gpt-6-luna.jsonl" in copied, "native pilot luna shard missing"
assert "prompts_native_pilot_claude-haiku-5-5.jsonl" in copied, "native pilot haiku shard missing"

bundle = OUT / "data" / "kaggle_datasets" / "thermo-agent-code" / "kaggle_bundle.zip"
assert bundle.is_file(), "kaggle_bundle.zip missing from HF snapshot"
shutil.copy2(bundle, UPLOAD / "kaggle_bundle.zip")

corpus_dir = OUT / "data" / "kaggle_datasets" / "thermo-agent-corpus"
if corpus_dir.is_dir():
    for src in sorted(corpus_dir.glob("*.zip")):
        shutil.copy2(src, UPLOAD / src.name)

readme = OUT / "README.md"
if readme.is_file():
    shutil.copy2(readme, UPLOAD / "README.md")

(UPLOAD / "dataset-metadata.json").write_text(json.dumps({
    "id": "rezanadimi/thermo-agent-data",
    "title": "thermo-agent data (HF mirror)",
    "licenses": [{"name": "MIT"}],
}, indent=1))
print(f"flat upload prepared: {len(copied)} prompt files, "
      f"{sum(p.stat().st_size for p in UPLOAD.iterdir())/1e6:.1f} MB", flush=True)

run_env = os.environ.copy()
run_env["KAGGLE_USERNAME"] = "rezanadimi"
proc = subprocess.run(
    [sys.executable, "-m", "kaggle", "datasets", "version", "-p", str(UPLOAD),
     "-m", "refresh from HF: native pilot shards", "-r", "skip"],
    env=run_env, capture_output=True, text=True)
print("kaggle version stdout:", proc.stdout[-3000:], flush=True)
print("kaggle version stderr:", proc.stderr[-3000:], flush=True)
if proc.returncode != 0:
    raise SystemExit(f"dataset version failed with code {proc.returncode}")
print("published new version of rezanadimi/thermo-agent-data", flush=True)
